In [1]:
user_history = {}
receiver_history = {}

print("Real-time histories initialized.")

Real-time histories initialized.


In [2]:
def generate_realtime_features(transaction):
    
    user = transaction["nameOrig"]
    receiver = transaction["nameDest"]
    step = transaction["step"]
    amount = transaction["amount"]
    transaction_type = transaction["type"]

    # -----------------------------
    # Sender history
    # -----------------------------
    
    user_hist = user_history.get(user, [])

    user_transaction_count_before = len(user_hist)

    if len(user_hist) > 0:
        previous_transaction_amount = user_hist[-1]["amount"]
        previous_step = user_hist[-1]["step"]

        time_since_previous_transaction = (
            step - previous_step
        )

        previous_average_amount = np.mean(
            [x["amount"] for x in user_hist]
        )

    else:
        previous_transaction_amount = np.nan
        time_since_previous_transaction = np.nan
        previous_average_amount = np.nan

    if pd.notna(previous_average_amount):
        amount_deviation = (
            amount - previous_average_amount
        )

        amount_to_previous_average = (
            amount / previous_average_amount
            if previous_average_amount != 0
            else 0
        )

    else:
        amount_deviation = np.nan
        amount_to_previous_average = np.nan


    # -----------------------------
    # Receiver history
    # -----------------------------

    receiver_hist = receiver_history.get(
        receiver,
        []
    )

    receiver_transaction_count_before = len(
        receiver_hist
    )

    if len(receiver_hist) > 0:

        receiver_previous_amount = (
            receiver_hist[-1]["amount"]
        )

        first_receiver_step = (
            receiver_hist[0]["step"]
        )

        receiver_transaction_frequency = (
            len(receiver_hist)
            /
            max(
                step - first_receiver_step,
                1
            )
        )

    else:

        receiver_previous_amount = np.nan
        receiver_transaction_frequency = np.nan


    # -----------------------------
    # Sender transaction behaviour
    # -----------------------------

    user_transfer_count_before = sum(
        1
        for x in user_hist
        if x["type"] == "TRANSFER"
    )

    user_cashout_count_before = sum(
        1
        for x in user_hist
        if x["type"] == "CASH_OUT"
    )


    # -----------------------------
    # Transaction velocity
    # -----------------------------

    transaction_velocity = len(user_hist)


    # -----------------------------
    # Balance features
    # -----------------------------

    oldbalance_org = transaction["oldbalanceOrg"]
    newbalance_orig = transaction["newbalanceOrig"]

    oldbalance_dest = transaction["oldbalanceDest"]
    newbalance_dest = transaction["newbalanceDest"]

    balance_depletion = (
        oldbalance_org - newbalance_orig
    )

    if oldbalance_org > 0:

        amount_to_balance_ratio = (
            amount / oldbalance_org
        )

    else:

        amount_to_balance_ratio = 0


    # -----------------------------
    # Create feature dictionary
    # -----------------------------

    features = {

        "step": step,
        "amount": amount,

        "oldbalanceOrg": oldbalance_org,
        "newbalanceOrig": newbalance_orig,

        "oldbalanceDest": oldbalance_dest,
        "newbalanceDest": newbalance_dest,

        "isFlaggedFraud":
            transaction.get(
                "isFlaggedFraud",
                0
            ),

        "user_transaction_count_before":
            user_transaction_count_before,

        "previous_transaction_amount":
            previous_transaction_amount,

        "time_since_previous_transaction":
            time_since_previous_transaction,

        "previous_average_amount":
            previous_average_amount,

        "amount_deviation":
            amount_deviation,

        "amount_to_previous_average":
            amount_to_previous_average,

        "balance_depletion":
            balance_depletion,

        "amount_to_balance_ratio":
            amount_to_balance_ratio,

        "receiver_transaction_count_before":
            receiver_transaction_count_before,

        "receiver_previous_amount":
            receiver_previous_amount,

        "receiver_transaction_frequency":
            receiver_transaction_frequency,

        "user_transfer_count_before":
            user_transfer_count_before,

        "user_cashout_count_before":
            user_cashout_count_before,

        "transaction_velocity":
            transaction_velocity
    }


    # -----------------------------
    # Update sender history
    # -----------------------------

    user_hist.append({
        "step": step,
        "amount": amount,
        "type": transaction_type
    })

    user_history[user] = user_hist


    # -----------------------------
    # Update receiver history
    # -----------------------------

    receiver_hist.append({
        "step": step,
        "amount": amount
    })

    receiver_history[receiver] = receiver_hist


    return features

In [3]:
import numpy as np
import pandas as pd

In [4]:
test_transaction = {
    "step": 1,
    "type": "TRANSFER",
    "amount": 1000,
    "nameOrig": "C100",
    "oldbalanceOrg": 5000,
    "newbalanceOrig": 4000,
    "nameDest": "M100",
    "oldbalanceDest": 0,
    "newbalanceDest": 1000,
    "isFlaggedFraud": 0
}

In [5]:
features = generate_realtime_features(
    test_transaction
)

print(features)

{'step': 1, 'amount': 1000, 'oldbalanceOrg': 5000, 'newbalanceOrig': 4000, 'oldbalanceDest': 0, 'newbalanceDest': 1000, 'isFlaggedFraud': 0, 'user_transaction_count_before': 0, 'previous_transaction_amount': nan, 'time_since_previous_transaction': nan, 'previous_average_amount': nan, 'amount_deviation': nan, 'amount_to_previous_average': nan, 'balance_depletion': 1000, 'amount_to_balance_ratio': 0.2, 'receiver_transaction_count_before': 0, 'receiver_previous_amount': nan, 'receiver_transaction_frequency': nan, 'user_transfer_count_before': 0, 'user_cashout_count_before': 0, 'transaction_velocity': 0}


In [6]:
test_transaction_2 = {
    "step": 2,
    "type": "CASH_OUT",
    "amount": 500,
    "nameOrig": "C100",
    "oldbalanceOrg": 4000,
    "newbalanceOrig": 3500,
    "nameDest": "M100",
    "oldbalanceDest": 1000,
    "newbalanceDest": 1500,
    "isFlaggedFraud": 0
}

In [7]:
features_2 = generate_realtime_features(
    test_transaction_2
)

print(features_2)

{'step': 2, 'amount': 500, 'oldbalanceOrg': 4000, 'newbalanceOrig': 3500, 'oldbalanceDest': 1000, 'newbalanceDest': 1500, 'isFlaggedFraud': 0, 'user_transaction_count_before': 1, 'previous_transaction_amount': 1000, 'time_since_previous_transaction': 1, 'previous_average_amount': np.float64(1000.0), 'amount_deviation': np.float64(-500.0), 'amount_to_previous_average': np.float64(0.5), 'balance_depletion': 500, 'amount_to_balance_ratio': 0.125, 'receiver_transaction_count_before': 1, 'receiver_previous_amount': 1000, 'receiver_transaction_frequency': 1.0, 'user_transfer_count_before': 1, 'user_cashout_count_before': 0, 'transaction_velocity': 1}


In [8]:
TRANSACTION_TYPES = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER"
]


def add_transaction_type_features(
    features,
    transaction_type
):

    for t in TRANSACTION_TYPES:

        features[f"type_{t}"] = (
            1 if transaction_type == t else 0
        )

    return features

In [9]:
final_features = add_transaction_type_features(
    features_2,
    test_transaction_2["type"]
)

print(final_features)

{'step': 2, 'amount': 500, 'oldbalanceOrg': 4000, 'newbalanceOrig': 3500, 'oldbalanceDest': 1000, 'newbalanceDest': 1500, 'isFlaggedFraud': 0, 'user_transaction_count_before': 1, 'previous_transaction_amount': 1000, 'time_since_previous_transaction': 1, 'previous_average_amount': np.float64(1000.0), 'amount_deviation': np.float64(-500.0), 'amount_to_previous_average': np.float64(0.5), 'balance_depletion': 500, 'amount_to_balance_ratio': 0.125, 'receiver_transaction_count_before': 1, 'receiver_previous_amount': 1000, 'receiver_transaction_frequency': 1.0, 'user_transfer_count_before': 1, 'user_cashout_count_before': 0, 'transaction_velocity': 1, 'type_CASH_IN': 0, 'type_CASH_OUT': 1, 'type_DEBIT': 0, 'type_PAYMENT': 0, 'type_TRANSFER': 0}


In [10]:
FEATURE_COLUMNS = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",

    "user_transaction_count_before",
    "previous_transaction_amount",
    "time_since_previous_transaction",
    "previous_average_amount",
    "amount_deviation",
    "amount_to_previous_average",
    "balance_depletion",
    "amount_to_balance_ratio",

    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",

    "user_transfer_count_before",
    "user_cashout_count_before",

    "transaction_velocity",

    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]

final_feature_df = pd.DataFrame(
    [final_features]
)

final_feature_df = final_feature_df[
    FEATURE_COLUMNS
]

final_feature_df = final_feature_df.fillna(0)

print("Number of features:", final_feature_df.shape[1])
print("Shape:", final_feature_df.shape)

Number of features: 26
Shape: (1, 26)


In [12]:
from xgboost import XGBClassifier

MODEL_FILE = "../models_saved/behaviour_aware_xgboost.json"

model = XGBClassifier()
model.load_model(MODEL_FILE)

print("Behaviour-Aware XGBoost loaded successfully.")

Behaviour-Aware XGBoost loaded successfully.


In [13]:
def predict_realtime_transaction(transaction):

    # --------------------------------
    # 1. Generate behavioural features
    # --------------------------------

    features = generate_realtime_features(
        transaction
    )

    # --------------------------------
    # 2. Add transaction type features
    # --------------------------------

    features = add_transaction_type_features(
        features,
        transaction["type"]
    )

    # --------------------------------
    # 3. Create DataFrame
    # --------------------------------

    input_df = pd.DataFrame(
        [features]
    )

    # --------------------------------
    # 4. Ensure exact feature order
    # --------------------------------

    input_df = input_df.reindex(
        columns=FEATURE_COLUMNS,
        fill_value=0
    )

    # --------------------------------
    # 5. Handle missing values
    # --------------------------------

    input_df = input_df.fillna(0)

    # --------------------------------
    # 6. Predict fraud probability
    # --------------------------------

    probability = model.predict_proba(
        input_df
    )[0, 1]

    # --------------------------------
    # 7. Classification
    # --------------------------------

    if probability >= 0.5:
        decision = "FRAUD"
    else:
        decision = "LEGITIMATE"

    # --------------------------------
    # 8. Risk level
    # --------------------------------

    if probability >= 0.8:
        risk = "HIGH"

    elif probability >= 0.5:
        risk = "MEDIUM"

    else:
        risk = "LOW"

    return {
        "fraud_probability": float(probability),
        "decision": decision,
        "risk_level": risk
    }

In [14]:
result = predict_realtime_transaction(
    test_transaction_2
)

print(result)

{'fraud_probability': 0.000816579326055944, 'decision': 'LEGITIMATE', 'risk_level': 'LOW'}


In [15]:
suspicious_transaction = {
    "step": 10,
    "type": "TRANSFER",
    "amount": 450000,
    "nameOrig": "C999999999",
    "oldbalanceOrg": 450500,
    "newbalanceOrig": 500,
    "nameDest": "M999999999",
    "oldbalanceDest": 1000,
    "newbalanceDest": 451000,
    "isFlaggedFraud": 0
}

In [16]:
suspicious_result = predict_realtime_transaction(
    suspicious_transaction
)

print(suspicious_result)

{'fraud_probability': 5.786254860140616e-06, 'decision': 'LEGITIMATE', 'risk_level': 'LOW'}


In [18]:
normal_transaction = {
    "step": 11,
    "type": "PAYMENT",
    "amount": 500,
    "nameOrig": "C888888888",
    "oldbalanceOrg": 5000,
    "newbalanceOrig": 4500,
    "nameDest": "M888888888",
    "oldbalanceDest": 10000,
    "newbalanceDest": 10500,
    "isFlaggedFraud": 0
}

In [19]:
normal_result = predict_realtime_transaction(
    normal_transaction
)

print(normal_result)

{'fraud_probability': 4.995009135200235e-07, 'decision': 'LEGITIMATE', 'risk_level': 'LOW'}


In [20]:
result = predict_realtime_transaction(
    test_transaction_2
)

print(result)

{'fraud_probability': 0.000348180765286088, 'decision': 'LEGITIMATE', 'risk_level': 'LOW'}
